# Install packages

In [1]:
!pip install lm_eval -q
!pip install lm-eval[openai] -q
!pip install lm-eval[anthropic] -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 1.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 243.3/243.3 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.0/104.0 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.1/111.1 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 44.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 8.5 MB/s eta 0:00:00
   ━

# lm_eval Import Check

In [ ]:
from lm_eval import api # to verify lm_eval is installed correctly

# (Optional) TODO: Status check of currently ran evals

# Loading Environment Variables

In [17]:
from dotenv import load_dotenv
import os

# Load .env file
load_dotenv()

# Pull the OPENROUTER_API_KEY into environment for subprocess & LM-Eval
api_key = os.getenv("OPENROUTER_API_KEY")

if not api_key:
    raise ValueError("OPENROUTER_API_KEY not found in .env!")

os.environ["OPENROUTER_API_KEY"] = api_key

print(f"Loaded OPENROUTER_API_KEY: {api_key[:10]}... (hidden)")


Loaded OPENROUTER_API_KEY: sk-or-v1-8... (hidden)


# OpenRouter Simple Chat Check

In [ ]:
import os
from openai import OpenAI

In [9]:
TASK_NAME    = "sysengbench-osq"
MODEL  = "openai/gpt-5"
TEMPERATURE  = 0.0
MAX_TOKENS   = 2000
SAMPLE_N     = 3          # 0 = judge ALL samples; else judge first N (for quick tests)

# # Paths (this notebook under: src/phase5_llm_as_a_judge/)
# PHASE4_ROOT  = Path("../phase4_inference/output") / TASK_NAME
# # PHASE5_ROOT  = Path(".") / TASK_NAME  # mirror structure in phase5
# PHASE5_ROOT  = Path(".") / f"{TASK_NAME}-llm-judge" # Append "-llm-judge" to keep Phase 5 artifacts separate and consistent
# PHASE5_ROOT.mkdir(parents=True, exist_ok=True)

# OpenRouter client
client = OpenAI(
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1"
)
print("OPENROUTER_API_KEY present:", bool(os.getenv("OPENROUTER_API_KEY")))

OPENROUTER_API_KEY present: True


In [10]:
completion = client.chat.completions.create(
    model=chatgpt_model,
    temperature=TEMPERATURE,
    max_tokens=MAX_TOKENS,
    messages=[
        {"role": "system", "content": "You are an expert evaluator in systems engineering education."},
        {"role": "user", "content": "What the neumber of letters in the longest word?"}
    ]
)
raw = (completion.choices[0].message.content or "").strip()

display(raw)

'The longest word in the English language is often cited as "pneumonoultramicroscopicsilicovolcanoconiosis," which has 45 letters. It refers to a type of lung disease caused by inhaling very fine silicate or quartz dust. However, there are even longer words in other contexts, such as chemical names, but they are not typically used in everyday language. If you have a specific context in mind, please let me know!'

# Local-chat-completions Test (May work if SysEngBench asks for 16+ tokens)

Previous problem when it wasn't returning proper JSONS was that I wasn't hitting the minimum 16 tokens as the current SysEngBench yaml only allowed for 10 tokens. 

Hypothesis: Set to 16 (or to be safe, 20) tokens in the yaml and this may work. Unconfirmed.

Will ened to remove eos_string more than liekly. Also still not sure where the api_key pulls from. I assume it is OPENAI since that's generally the default in lm-eval if not passed in as an arg.

In [15]:
import subprocess
import os

cmd = [
    "lm_eval",
    "--model", "local-chat-completions",
    "--model_args",
    (
        f"model=openai/{chatgpt_model},"
        "base_url=https://openrouter.ai/api/v1,"
        f"api_key={api_key},"
        "eos_string=<EOS>"
    ),
    "--limit", "3",
    "--include_path", "./",
    "--tasks", "sysengbench",
    "--apply_chat_template",
    "--verbosity", "DEBUG",        # ← added here
]

# Need to comment this out to not print the key.
print("Running command:")
print(" ".join(cmd))

result = subprocess.run(
    cmd,
    text=True,
    capture_output=True
)

print("\n=== STDOUT ===")
print(result.stdout)

print("\n=== STDERR ===")
print(result.stderr)


Running command:
lm_eval --model local-chat-completions --model_args model=openai/gpt-4o-mini,base_url=https://openrouter.ai/api/v1,api_key=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,eos_string=<EOS> --limit 3 --include_path ./ --tasks sysengbench --apply_chat_template --verbosity DEBUG

=== STDOUT ===
Selected Tasks: ['sysengbench']


=== STDERR ===
2025-11-15:22:08:25 INFO     [__main__:348] Including path: ./
2025-11-15:22:08:40 WARNING  [__main__:369]  --limit SHOULD ONLY BE USED FOR TESTING.REAL METRICS SHOULD NOT BE COMPUTED USING LIMIT.
2025-11-15:22:08:40 INFO     [evaluator:202] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
2025-11-15:22:08:40 INFO     [evaluator:240] Initializing local-chat-completions model, with arguments: {'model': 'openai/gpt-4o-mini', 'base_url': 'https://openrouter.ai/api/v1',
        'api_key': 'sk-or-v1-8126d117544792e8030b90b128050a27d94df

# Patch Script, Add OpenRouter Model Adapter to Lm-eval

In [4]:
import importlib
import sys
from pathlib import Path

# Path where we want to create the custom backend
backend_dir = Path(sys.prefix) / "Lib" / "site-packages" / "lm_eval" / "models"
backend_file = backend_dir / "openrouter_chat_completions.py"

# Write the custom backend
backend_code = """
import os
import requests as http_requests
from lm_eval.api.model import LM
from lm_eval.api.registry import register_model

@register_model("openrouter-chat-completions")
class OpenRouterChatCompletions(LM):

    def __init__(self, model, base_url="https://openrouter.ai/api/v1",
                 api_key=None, **kwargs):
        super().__init__()
        self.model = model
        self.base_url = base_url.rstrip("/")
        # Load API key from environment if not provided
        self.api_key = api_key or os.environ.get("OPENROUTER_API_KEY")
        if not self.api_key:
            raise ValueError("OpenRouter API key not found. Set OPENROUTER_API_KEY environment variable or pass api_key parameter.")
        self.kwargs = kwargs

    @property
    def tokenizer_name(self):
        # Required for chat template selection
        return self.model

    @staticmethod
    def from_config(config):
        return OpenRouterChatCompletions(
            model=config.get("model"),
            base_url=config.get("base_url", "https://openrouter.ai/api/v1"),
            api_key=config.get("api_key"),
            **config,
        )

    def generate_until(self, requests):
        import json
        outputs = []
        for req in requests:
            # req is an Instance object with args property
            # args is a tuple: (context, gen_kwargs)
            context, gen_kwargs = req.args
        
            # IMPORTANT: Override the arguments attribute to make it hashable
            # The evaluator tries to hash arguments[0] which is the context (a list for chat)
            # We need to convert the list to a JSON string for hashing
            if isinstance(context, list):
                # Store original context
                original_arguments = req.arguments
                # Create hashable version - convert list to JSON string
                req.arguments = (json.dumps(context, sort_keys=True), gen_kwargs)
            
            # Extract generation parameters from gen_kwargs
            stops = gen_kwargs.get("until", None)
            max_tokens = gen_kwargs.get("max_gen_toks", 256)
            temperature = gen_kwargs.get("temperature", 0)
            
            # Ensure max_tokens meets minimum API requirements (some providers require >= 16)
            original_max_tokens = max_tokens
            max_tokens = max(max_tokens, 16)
            if original_max_tokens < 16:
                print(f"⚠️  Adjusted max_tokens from {original_max_tokens} to {max_tokens} (OpenRouter/Azure minimum requirement)")

            payload = {
                "model": self.model,
                "messages": context if isinstance(context, list) else [{"role": "user", "content": context}],
                "max_tokens": max_tokens,
                "temperature": temperature,
            }
            
            # Only include stop if it's not None and not an empty list
            if stops and len(stops) > 0:
                payload["stop"] = stops

            headers = {
                "Authorization": f"Bearer {self.api_key}",
                "Content-Type": "application/json",
                "HTTP-Referer": "lm-evaluation-harness",
            }

            try:
                r = http_requests.post(
                    f"{self.base_url}/chat/completions",
                    json=payload,
                    headers=headers,
                    timeout=60,
                )
                r.raise_for_status()
                data = r.json()
            except Exception as e:
                # Add debugging info
                print(f"Error calling OpenRouter API: {e}")
                print(f"Payload: {payload}")
                print(f"Response: {r.text if 'r' in locals() else 'No response'}")
                raise

            text = data["choices"][0]["message"]["content"]
            outputs.append(text)

        return outputs

    def loglikelihood(self, requests):
        raise NotImplementedError("OpenRouter does not support logprobs.")

    def loglikelihood_rolling(self, requests):
        raise NotImplementedError("OpenRouter does not support logprobs.")
    
    def apply_chat_template(self, messages, add_generation_prompt=True):
        # For chat APIs, return messages as-is since the API handles formatting.
        # The add_generation_prompt parameter is accepted but ignored for API-based models.
        if isinstance(messages, list) and all(isinstance(m, dict) for m in messages):
            return messages
        if isinstance(messages, str):
            return [{"role": "user", "content": messages}]
        return messages
    
    def tok_encode(self, string, left_truncate_len=None, add_special_tokens=None, **kwargs):
        # For chat APIs, we don't need actual tokenization. 
        # Handle lists (chat messages) by converting to JSON string for hashing.
        if isinstance(string, list):
            import json
            return json.dumps(string, sort_keys=True)
        return string
"""

backend_file.write_text(backend_code)
print(f"✅ Created backend at: {backend_file}")

# Patch the models/__init__.py to import our backend
init_file = backend_dir / "__init__.py"
init_content = init_file.read_text()

# Check if our import is already there
import_line = "from . import openrouter_chat_completions"
if import_line not in init_content:
    # Find where to add it (after other imports)
    lines = init_content.split('\n')
    # Find the last "from . import" line
    last_import_idx = 0
    for i, line in enumerate(lines):
        if line.strip().startswith('from . import'):
            last_import_idx = i
    
    # Insert our import after the last one
    lines.insert(last_import_idx + 1, import_line)
    init_file.write_text('\n'.join(lines))
    print("✅ Patched models/__init__.py")
else:
    print("✅ Import already exists in models/__init__.py")

# Now verify the model is registered
from lm_eval.api.registry import MODEL_REGISTRY
print("\nRegistered models containing 'openrouter':")
for name in MODEL_REGISTRY:
    if 'openrouter' in name.lower():
        print(f"  - {name}")

if 'openrouter-chat-completions' in MODEL_REGISTRY:
    print("\n✅ openrouter-chat-completions is registered!")
else:
    print("\n❌ openrouter-chat-completions NOT found in registry")
    print("You may need to restart your kernel.")

🔍 Detecting lm_eval installation...
📂 lm_eval located at: c:\Users\rabel\Desktop\dissertation\.venv\Lib\site-packages\lm_eval
📂 models directory: c:\Users\rabel\Desktop\dissertation\.venv\Lib\site-packages\lm_eval\models
📝 Wrote backend: c:\Users\rabel\Desktop\dissertation\.venv\Lib\site-packages\lm_eval\models\openrouter_chat_completions.py
✔️ models/__init__.py already contains backend import.

🔍 Checking MODEL_REGISTRY keys:
['local-completions', 'local-chat-completions', 'openai-completions', 'openai-chat-completions', 'anthropic-completions', 'anthropic-chat', 'anthropic-chat-completions', 'dummy', 'gguf', 'ggml', 'hf-auto', 'hf', 'huggingface', 'hf-audiolm-qwen', 'steered', 'hf-multimodal', 'watsonx_llm', 'mamba_ssm', 'nemo_lm', 'neuronx', 'ipex', 'openvino', 'sglang', 'sglang-generate', 'textsynth', 'vllm', 'vllm-vlm', 'openrouter-chat-completions']
🎉 SUCCESS: backend registered!


In [1]:
# Check that the openrouter-chat-completions.py was successful
import lm_eval.models
from lm_eval.api.registry import MODEL_REGISTRY

print(MODEL_REGISTRY.keys())


dict_keys(['local-completions', 'local-chat-completions', 'openai-completions', 'openai-chat-completions', 'anthropic-completions', 'anthropic-chat', 'anthropic-chat-completions', 'dummy', 'gguf', 'ggml', 'hf-auto', 'hf', 'huggingface', 'hf-audiolm-qwen', 'steered', 'hf-multimodal', 'watsonx_llm', 'mamba_ssm', 'nemo_lm', 'neuronx', 'ipex', 'openvino', 'sglang', 'sglang-generate', 'textsynth', 'vllm', 'vllm-vlm', 'openrouter-chat-completions'])


# Open Router Evaluations

## Single OpenRouter

In [1]:
!lm_eval \
  --model openrouter-chat-completions \
  --model_args "model=openai/gpt-4.1" \
  --include_path ./ \
  --tasks sysengbench \
  --num_fewshot 0 \
  --log_samples \
  --limit 4 \
  --output output/sysengbench \
  --apply_chat_template \
  --verbosity DEBUG

Selected Tasks: ['sysengbench', 'sysengbench-a']
⚠️  Adjusted max_tokens from 10 to 16 (OpenRouter/Azure minimum requirement)
⚠️  Adjusted max_tokens from 10 to 16 (OpenRouter/Azure minimum requirement)
⚠️  Adjusted max_tokens from 10 to 16 (OpenRouter/Azure minimum requirement)
⚠️  Adjusted max_tokens from 10 to 16 (OpenRouter/Azure minimum requirement)
⚠️  Adjusted max_tokens from 10 to 16 (OpenRouter/Azure minimum requirement)
⚠️  Adjusted max_tokens from 10 to 16 (OpenRouter/Azure minimum requirement)
⚠️  Adjusted max_tokens from 10 to 16 (OpenRouter/Azure minimum requirement)
⚠️  Adjusted max_tokens from 10 to 16 (OpenRouter/Azure minimum requirement)
openrouter-chat-completions (model=openai/gpt-4.1), gen_kwargs: (None), limit: 4.0, num_fewshot: 0, batch_size: 1
|    Tasks    |Version|   Filter   |n-shot|  Metric   |   |Value|   |Stderr|
|-------------|------:|------------|-----:|-----------|---|----:|---|-----:|
|sysengbench  |      1|strict-match|     0|exact_match|↑  |    1|± 

2025-11-16:01:23:48 INFO     [__main__:348] Including path: ./
2025-11-16:01:23:57 WARNING  [__main__:369]  --limit SHOULD ONLY BE USED FOR TESTING.REAL METRICS SHOULD NOT BE COMPUTED USING LIMIT.
2025-11-16:01:23:57 INFO     [evaluator:202] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
2025-11-16:01:23:57 INFO     [evaluator:240] Initializing openrouter-chat-completions model, with arguments: {'model': 'openai/gpt-4.1'}
2025-11-16:01:24:01 INFO     [evaluator:305] sysengbench-a: Using gen_kwargs: {'until': [], 'max_gen_toks': 10, 'temperature': 0.0}
2025-11-16:01:24:01 WARNING  [evaluator:324] Overwriting default num_fewshot of sysengbench-a from None to 0
2025-11-16:01:24:01 INFO     [evaluator:305] sysengbench: Using gen_kwargs: {'until': [], 'max_gen_toks': 10, 'temperature': 0.0}
2025-11-16:01:24:01 WARNING  [evaluator:324] Overwriting default num_fewshot of sysengbench from None to 0
2025-11-16:01:2

## 1 Model, Multiple Tasks

In [4]:
import subprocess
import sys

# Model and tasks to run
model = "google/gemini-2.5-flash"
tasks = ["sysengbench", "sysengbench-a"]

# Auto-generate folder name from model: replace / with __
model_folder = model.replace("/", "__")

print(f"Model: {model}")
print(f"Folder name: {model_folder}")

# Run each task separately with its own output directory
for task in tasks:
    output_dir = f"output/{task}"
    
    print(f"\n{'='*80}")
    print(f"Running {model} on {task}")
    print(f"Output: {output_dir}")
    print(f"{'='*80}\n")
    
    cmd = [
        "lm_eval",
        "--model", "openrouter-chat-completions",
        "--model_args", f"model={model}",
        "--include_path", "./",
        "--tasks", task,
        "--num_fewshot", "0",
        "--log_samples",
        "--limit", "4",
        "--output", output_dir,
        "--apply_chat_template",
        "--verbosity", "DEBUG"
    ]
    
    # Use encoding='utf-8' and errors='replace' to handle Unicode characters
    result = subprocess.run(cmd, capture_output=True, text=True, encoding='utf-8', errors='replace')
    print(result.stdout)
    if result.returncode != 0:
        print("STDERR:", result.stderr)

Model: google/gemini-2.5-flash
Folder name: google__gemini-2.5-flash

Running google/gemini-2.5-flash on sysengbench
Output: output/sysengbench

Selected Tasks: ['sysengbench']
⚠️  Adjusted max_tokens from 10 to 16 (OpenRouter/Azure minimum requirement)
⚠️  Adjusted max_tokens from 10 to 16 (OpenRouter/Azure minimum requirement)
⚠️  Adjusted max_tokens from 10 to 16 (OpenRouter/Azure minimum requirement)
⚠️  Adjusted max_tokens from 10 to 16 (OpenRouter/Azure minimum requirement)
openrouter-chat-completions (model=google/gemini-2.5-flash), gen_kwargs: (None), limit: 4.0, num_fewshot: 0, batch_size: 1
|   Tasks   |Version|   Filter   |n-shot|  Metric   |   |Value|   |Stderr|
|-----------|------:|------------|-----:|-----------|---|----:|---|-----:|
|sysengbench|      1|strict-match|     0|exact_match|↑  |    1|±  |     0|



Running google/gemini-2.5-flash on sysengbench-a
Output: output/sysengbench-a

Selected Tasks: ['sysengbench']
⚠️  Adjusted max_tokens from 10 to 16 (OpenRouter/Azu

## Multiple Models, Multiple Tasks

If needed to limit below:

            "--limit", "2",

In [ ]:
import subprocess
import sys

# Define multiple models to test
models = [
    # "openai/gpt-5",
    "openai/gpt-4.1",
    "google/gemini-2.5-flash",
    "anthropic/claude-sonnet-4.5",
]

# Define tasks (use hyphen format to match existing structure)
tasks = ["sysengbench", "sysengbench-a", "sysengbench-b", "sysengbench-c", "sysengbench-d", "sysengbench-osq"]


In [5]:
import subprocess
import sys

# Define multiple models to test
models = [
    # "openai/gpt-5",
    # "openai/gpt-4.1",
    # "google/gemini-2.5-flash",
    "anthropic/claude-sonnet-4.5",
]

# Define tasks (use hyphen format to match existing structure)
# tasks = ["sysengbench", "sysengbench-a", "sysengbench-b", "sysengbench-c", "sysengbench-d", "sysengbench-osq"]

# tasks = ["sysengbench-b", "sysengbench-c","sysengbench-osq"]
# tasks =["sysengbench-d"]
tasks =["sysengbench-osq"]

# claud on b,c, osq
# gemini on d


In [6]:
# Loop through each model and task
for model in models:
    # Auto-generate folder name from model: replace / with __
    model_folder = model.replace("/", "__")
    
    print(f"\n{'#'*80}")
    print(f"# MODEL: {model} ({model_folder})")
    print(f"{'#'*80}\n")
    
    for task in tasks:
        output_dir = f"output/{task}"
        
        print(f"\n{'='*80}")
        print(f"Running {model} on {task}")
        print(f"Output: {output_dir}")
        print(f"{'='*80}\n")
        
        cmd = [
            "lm_eval",
            "--model", "openrouter-chat-completions",
            "--model_args", f"model={model}",
            "--include_path", "./",
            "--tasks", task,
            "--num_fewshot", "0",
            "--log_samples",
            "--output", output_dir,
            "--apply_chat_template",
            "--batch_size", "1"
        ]
        
        # Use encoding='utf-8' and errors='replace' to handle Unicode characters
        result = subprocess.run(cmd, capture_output=True, text=True, encoding='utf-8', errors='replace')
        print(result.stdout)
        if result.returncode != 0:
            print("STDERR:", result.stderr)
            print(f"❌ Failed on {model} / {task}")
            # Continue to next task instead of stopping
            continue
        else:
            print(f"✅ Completed {model} / {task}")

print("\n" + "#"*80)
print("# ALL EVALUATIONS COMPLETE!")
print("#"*80)


################################################################################
# MODEL: anthropic/claude-sonnet-4.5 (anthropic__claude-sonnet-4.5)
################################################################################


Running anthropic/claude-sonnet-4.5 on sysengbench-osq
Output: output/sysengbench-osq

openrouter-chat-completions (model=anthropic/claude-sonnet-4.5), gen_kwargs: (None), limit: None, num_fewshot: 0, batch_size: 1
|     Tasks     |Version|Filter|n-shot|  Metric   |   |Value |   |Stderr|
|---------------|------:|------|-----:|-----------|---|-----:|---|-----:|
|sysengbench-osq|      1|none  |     0|exact_match|↑  |0.0651|±  |0.0085|


✅ Completed anthropic/claude-sonnet-4.5 / sysengbench-osq

################################################################################
# ALL EVALUATIONS COMPLETE!
################################################################################
openrouter-chat-completions (model=anthropic/claude-sonnet-4.5), gen_kwargs: (N

## NEXT TODO: Add the ability to paralleize the processes. (UNVERIFIED)

In [ ]:
import subprocess
import sys
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime

# Configuration
MAX_CONCURRENT_WORKERS = 4  # Set this to control how many evaluations run in parallel

# Define multiple models to test
models = [
    "openai/gpt-5",
    "openai/gpt-4o-mini",
    "anthropic/claude-3.5-sonnet",
    "google/gemini-2.0-flash-exp",
    "mistralai/mistral-large-2411",
    "deepseek/deepseek-chat"
]

# Define tasks (use hyphen format to match existing structure)
tasks = ["sysengbench", "sysengbench-a", "sysengbench-b", "sysengbench-c", "sysengbench-d", "sysengbench-osq"]

def run_evaluation(model, task):
    """Run a single evaluation and return the result."""
    model_folder = model.replace("/", "__")
    output_dir = f"output/{task}"
    
    start_time = datetime.now()
    print(f"[{start_time.strftime('%H:%M:%S')}] 🚀 Starting: {model} on {task}")
    
    cmd = [
        "lm_eval",
        "--model", "openrouter-chat-completions",
        "--model_args", f"model={model}",
        "--include_path", "./",
        "--tasks", task,
        "--num_fewshot", "0",
        "--log_samples",
        "--output", output_dir,
        "--apply_chat_template",
        "--batch_size", "1"
    ]
    
    try:
        result = subprocess.run(
            cmd, 
            capture_output=True, 
            text=True, 
            encoding='utf-8', 
            errors='replace',
            timeout=3600  # 1 hour timeout per evaluation
        )
        
        end_time = datetime.now()
        duration = (end_time - start_time).total_seconds()
        
        if result.returncode == 0:
            return {
                'model': model,
                'task': task,
                'status': 'success',
                'duration': duration,
                'output': result.stdout
            }
        else:
            return {
                'model': model,
                'task': task,
                'status': 'failed',
                'duration': duration,
                'error': result.stderr
            }
    except subprocess.TimeoutExpired:
        return {
            'model': model,
            'task': task,
            'status': 'timeout',
            'duration': 3600,
            'error': 'Evaluation timed out after 1 hour'
        }
    except Exception as e:
        return {
            'model': model,
            'task': task,
            'status': 'error',
            'duration': 0,
            'error': str(e)
        }

# Create list of all jobs (model, task combinations)
jobs = [(model, task) for model in models for task in tasks]
total_jobs = len(jobs)

print(f"\n{'#'*80}")
print(f"# PARALLEL EVALUATION")
print(f"# Total jobs: {total_jobs} ({len(models)} models × {len(tasks)} tasks)")
print(f"# Concurrent workers: {MAX_CONCURRENT_WORKERS}")
print(f"{'#'*80}\n")

# Run evaluations in parallel
results = []
completed = 0

with ThreadPoolExecutor(max_workers=MAX_CONCURRENT_WORKERS) as executor:
    # Submit all jobs
    future_to_job = {executor.submit(run_evaluation, model, task): (model, task) 
                     for model, task in jobs}
    
    # Process completed jobs as they finish
    for future in as_completed(future_to_job):
        model, task = future_to_job[future]
        try:
            result = future.result()
            results.append(result)
            completed += 1
            
            # Print status
            if result['status'] == 'success':
                print(f"[{completed}/{total_jobs}] ✅ {result['model']} / {result['task']} "
                      f"({result['duration']:.1f}s)")
            elif result['status'] == 'timeout':
                print(f"[{completed}/{total_jobs}] ⏱️  {result['model']} / {result['task']} "
                      f"(TIMEOUT)")
            else:
                print(f"[{completed}/{total_jobs}] ❌ {result['model']} / {result['task']} "
                      f"(FAILED)")
                
        except Exception as e:
            print(f"[{completed}/{total_jobs}] ⚠️  Exception for {model} / {task}: {e}")

# Summary
print(f"\n{'#'*80}")
print(f"# EVALUATION SUMMARY")
print(f"{'#'*80}")

success_count = sum(1 for r in results if r['status'] == 'success')
failed_count = sum(1 for r in results if r['status'] == 'failed')
timeout_count = sum(1 for r in results if r['status'] == 'timeout')
error_count = sum(1 for r in results if r['status'] == 'error')

print(f"✅ Successful: {success_count}/{total_jobs}")
print(f"❌ Failed: {failed_count}/{total_jobs}")
print(f"⏱️  Timeout: {timeout_count}/{total_jobs}")
print(f"⚠️  Errors: {error_count}/{total_jobs}")

if failed_count > 0 or timeout_count > 0 or error_count > 0:
    print(f"\n{'='*80}")
    print("FAILED/TIMEOUT/ERROR JOBS:")
    print(f"{'='*80}")
    for r in results:
        if r['status'] != 'success':
            print(f"  - {r['model']} / {r['task']}: {r['status']}")
            if 'error' in r:
                print(f"    Error: {r['error'][:200]}")

print(f"\n{'#'*80}")
print(f"# ALL EVALUATIONS COMPLETE!")
print(f"{'#'*80}")